# TP 01 : Analyse de Variance (ANOVA) à Deux Facteurs Croisés & Post-Hoc de Tukey
**Master 2 Biochimie Appliquée — Université M'Hamed Bougara de Boumerdès (UMBB)**  
*Enseignante : Dr. Sarra BENMOUMOU (Ph.D.)*

---

## Contexte Biologique :
Nous étudions l'effet hépatoprotecteur d'une molécule antioxydante naturelle (polyphénol) administrée à deux doses (50 et 100 mg/kg) vs Véhicule témoin, chez des souris saines (WT) et des souris modèles du diabète de type 2 (`db_db`).  
La variable d'intérêt est le taux hépatique de **Malondialdéhyde (MDA, $\mu$mol/g)**, marqueur de référence de la peroxydation lipidique membranaire.

## Objectifs Pédagogiques :
1. Visualiser et interpréter graphiquement les profils d'interaction (`interaction_plot`).
2. Calculer intégralement la table ANOVA à 2 facteurs : Sommes des Carrés ($SS$), Carrés Moyens ($MS$), et statistiques $F$.
3. Vérifier les postulats de validité : Normalité des résidus (Shapiro-Wilk) et Homoscédasticité (Levene).
4. Réaliser le test post-hoc de comparaisons multiples de **Tukey HSD**.
5. Conclure sur la sélectivité biologique de la molécule.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 1. Chargement du jeu de données expérimental
df = pd.read_csv('../datasets/peroxydation_mda_diabete.csv')
print(f"Effectif total : N = {len(df)} souris")
display(df.head(10))

## Étape 1 : Tracé et Analyse du Profil d'Interaction
Observons si les profils de réponse à la dose sont parallèles (additivité) ou sécants/croisés (interaction).

In [ ]:
plt.figure(figsize=(9, 5), dpi=120)

# Moyennes et erreurs par sous-groupe
mean_data = df.groupby(['Traitement', 'Genotype'])['MDA_nmol_mg_prot'].agg(['mean', 'sem']).reset_index()

sns.lineplot(
    data=df,
    x='Traitement',
    y='MDA_nmol_mg_prot',
    hue='Genotype',
    marker='o',
    markersize=9,
    linewidth=2.5,
    errorbar='se',
    palette={'WT': '#1A365D', 'db_db': '#D97706'}
)

plt.title("Profil d'Interaction : Traitement x Génotype (MDA Hépatique)", fontsize=13, fontweight='bold', color='#1A365D')
plt.xlabel("Dose de Polyphénol", fontweight='bold')
plt.ylabel("MDA Moyen ± SEM (nmol/mg prot)", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Étape 2 : Calcul Complet de la Table ANOVA à Deux Facteurs Croisés

In [ ]:
N = len(df)
grand_mean = df['MDA_nmol_mg_prot'].mean()

I = df['Traitement'].nunique()  # 3 traitements
J = df['Genotype'].nunique()    # 2 génotypes
K = N // (I * J)                # 5 souris par lot

# Sommes des Carrés (SS)
SS_total = ((df['MDA_nmol_mg_prot'] - grand_mean)**2).sum()

means_A = df.groupby('Traitement')['MDA_nmol_mg_prot'].mean()
SS_A = (J * K) * ((means_A - grand_mean)**2).sum()

means_B = df.groupby('Genotype')['MDA_nmol_mg_prot'].mean()
SS_B = (I * K) * ((means_B - grand_mean)**2).sum()

means_AB = df.groupby(['Traitement', 'Genotype'])['MDA_nmol_mg_prot'].mean()
SS_cell = K * ((means_AB - grand_mean)**2).sum()
SS_AB = SS_cell - SS_A - SS_B

SS_res = SS_total - SS_cell

# Degrés de liberté (ddl)
df_A = I - 1
df_B = J - 1
df_AB = (I - 1) * (J - 1)
df_res = N - (I * J)

# Carrés Moyens (MS = SS / ddl)
MS_A = SS_A / df_A
MS_B = SS_B / df_B
MS_AB = SS_AB / df_AB
MS_res = SS_res / df_res

# Statistique F de Fisher (MS / MS_res)
F_A = MS_A / MS_res
F_B = MS_B / MS_res
F_AB = MS_AB / MS_res

# p-values
p_A = 1 - stats.f.cdf(F_A, df_A, df_res)
p_B = 1 - stats.f.cdf(F_B, df_B, df_res)
p_AB = 1 - stats.f.cdf(F_AB, df_AB, df_res)

table_anova = pd.DataFrame({
    'Source de Variation': ['Traitement (A)', 'Génotype (B)', 'Interaction A x B', 'Résiduelle (Erreur)'],
    'ddl': [df_A, df_B, df_AB, df_res],
    'Somme Carrés (SS)': [round(SS_A, 2), round(SS_B, 2), round(SS_AB, 2), round(SS_res, 2)],
    'Carré Moyen (MS)': [round(MS_A, 2), round(MS_B, 2), round(MS_AB, 2), round(MS_res, 2)],
    'F_obs': [round(F_A, 2), round(F_B, 2), round(F_AB, 2), np.nan],
    'p_value': [f'{p_A:.4e}', f'{p_B:.4e}', f'{p_AB:.4e}', np.nan]
})

print("=== TABLE ANOVA A DEUX FACTEURS CROISES (CALCULÉE) ===")
display(table_anova)

## Étape 3 : Diagnostic des Postulats Paramétriques
1. **Normalité des résidus** : Test de Shapiro-Wilk.
2. **Homogénéité des variances** : Test de Levene.

In [ ]:
# Calcul des résidus du modèle
df['Groupe'] = df['Traitement'] + '_' + df['Genotype']
group_means = df.groupby('Groupe')['MDA_nmol_mg_prot'].transform('mean')
residus = df['MDA_nmol_mg_prot'] - group_means

# Test de Shapiro-Wilk sur les résidus
stat_shapiro, p_shapiro = stats.shapiro(residus)
print(f"Test de Shapiro-Wilk sur les résidus : W = {stat_shapiro:.4f}, p = {p_shapiro:.4f}")
if p_shapiro > 0.05:
    print("✅ Postulat de normalité des résidus VALIDÉ (p > 0.05)")
else:
    print("❌ Postulat de normalité violé")

# Test de Levene d'homoscédasticité
groupes_data = [group['MDA_nmol_mg_prot'].values for _, group in df.groupby('Groupe')]
stat_levene, p_levene = stats.levene(*groupes_data)
print(f"Test de Levene (Homoscédasticité) : W = {stat_levene:.4f}, p = {p_levene:.4f}")
if p_levene > 0.05:
    print("✅ Postulat d'homogénéité des variances VALIDÉ (p > 0.05)")
else:
    print("❌ Postulat d'homogénéité violé")

## Étape 4 : Test Post-Hoc de Comparaisons Multiples (Tukey HSD)
Puisque l'interaction est significative ($p_{AB} < 0.01$), analysons les comparaisons de Tukey pour identifier quelles paires diffèrent significativement.

In [ ]:
tukey_result = pairwise_tukeyhsd(
    endog=df['MDA_nmol_mg_prot'],
    groups=df['Groupe'],
    alpha=0.05
)

print("=== RÉSULTATS DU TEST POST-HOC DE TUKEY HSD ===")
print(tukey_result)

## Étape 5 : Interprétation Biologique & Conclusion Médicale :
1. **Pourquoi l'interaction Traitement $\times$ Génotype est-elle cliniquement déterminante ?**
   - Le polyphénol réduit-il le MDA chez les souris saines (WT) ?
   - Quel est le pourcentage de réduction observé chez les souris diabétiques ($db/db$) à la dose de 100 mg/kg ?
2. **Conclusion générale pour le rapport de laboratoire :**
   - Rédigez le paragraphe de conclusion au format standardisé international (ex : $F(2, 24) = \dots, p = \dots$).
